In [1]:
# Task 1: DCGAN - Synthetic Faces (Only 500 Images)

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
image_folder = "./dataset/celeba"  # <<< CHANGE THIS TO YOUR PATH
if not os.path.exists(image_folder):
    raise FileNotFoundError(f"Image folder not found: {image_folder}")

# Hyperparameters
latent_dim = 100
img_size = 64
channels = 3
batch_size = 64  # Reduced slightly for stability with small dataset
num_epochs = 50  # Increase epochs since data is limited
lr = 0.0002
beta1 = 0.5

# Transform
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # [-1, 1]
])

In [4]:
# Custom Dataset: Load only first 500 images
class LimitedImageDataset(Dataset):
    def __init__(self, root, transform=None, num_images=10000):
        self.root = root
        self.transform = transform
        # Get list of image files and limit to first `num_images`
        self.imgs = [
            f for f in os.listdir(root)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))
        ]
        # Sort for consistency
        self.imgs = sorted(self.imgs)[:num_images]  # First 500

        if len(self.imgs) == 0:
            raise ValueError(f"No images found in {root}")
        print(f"Loaded {len(self.imgs)} images (limit: {num_images}).")

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, self.imgs[idx])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, 0  # Dummy label

# Load dataset (only 500 images)
dataset = LimitedImageDataset(root=image_folder, transform=transform, num_images=10000)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

print(f"Dataset initialized with {len(dataset)} images.")

Loaded 10000 images (limit: 10000).
Dataset initialized with 10000 images.


In [5]:
# Generator Model (DCGAN)
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)


In [6]:
# Discriminator Model
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(channels, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x).view(-1, 1)


In [ ]:
# Initialize models
netG = Generator().to(device)
netD = Discriminator().to(device)

# Loss and optimizers
criterion = nn.BCELoss()
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))

# Fixed noise for consistent generation
fixed_noise = torch.randn(64, latent_dim, 1, 1, device=device)

# Training tracking
G_losses = []
D_losses = []
img_list = []

print("Starting DCGAN training on 500 images...")

for epoch in range(num_epochs):
    for i, (real_images, _) in enumerate(dataloader):
        real_images = real_images.to(device)
        b_size = real_images.size(0)

        # Labels
        real_labels = torch.ones(b_size, 1, device=device)
        fake_labels = torch.zeros(b_size, 1, device=device)

        ############################
        # Train Discriminator
        ############################
        netD.zero_grad()
        # Real images
        output = netD(real_images)
        lossD_real = criterion(output, real_labels)
        lossD_real.backward()

        # Fake images
        noise = torch.randn(b_size, latent_dim, 1, 1, device=device)
        fake_images = netG(noise)
        output = netD(fake_images.detach())
        lossD_fake = criterion(output, fake_labels)
        lossD_fake.backward()

        lossD = lossD_real + lossD_fake
        optimizerD.step()

        ############################
        # Train Generator
        ############################
        netG.zero_grad()
        output = netD(fake_images)
        lossG = criterion(output, real_labels)  # Fool discriminator
        lossG.backward()
        optimizerG.step()

        G_losses.append(lossG.item())
        D_losses.append(lossD.item())

        if i % 100 == 0:  # Print more frequently since dataset is small
            print(f"[Epoch {epoch+1}/{num_epochs}] [Batch {i}/{len(dataloader)}] "
                  f"Loss D: {lossD.item():.4f} Loss G: {lossG.item():.4f}")
        
        #print(f"Generator loss: {lossG.item()} Discriminator loss: {lossD.item()}")

    # Save generated image grid every 10 epochs
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            fake = netG(fixed_noise).cpu()
            img_grid = torchvision.utils.make_grid(fake, padding=2, normalize=True)
            img_list.append(img_grid)


Starting DCGAN training on 500 images...
[Epoch 1/50] [Batch 0/157] Loss D: 1.5870 Loss G: 2.3781
[Epoch 1/50] [Batch 100/157] Loss D: 1.0384 Loss G: 3.8065
[Epoch 2/50] [Batch 0/157] Loss D: 0.7806 Loss G: 2.5699
[Epoch 2/50] [Batch 100/157] Loss D: 0.5401 Loss G: 2.8923
[Epoch 3/50] [Batch 0/157] Loss D: 0.6562 Loss G: 1.5453
[Epoch 3/50] [Batch 100/157] Loss D: 0.6757 Loss G: 3.6373
[Epoch 4/50] [Batch 0/157] Loss D: 0.4096 Loss G: 2.6460
[Epoch 4/50] [Batch 100/157] Loss D: 0.4346 Loss G: 3.1489
[Epoch 5/50] [Batch 0/157] Loss D: 0.7390 Loss G: 4.5179
[Epoch 5/50] [Batch 100/157] Loss D: 0.3873 Loss G: 4.6587
[Epoch 6/50] [Batch 0/157] Loss D: 0.3987 Loss G: 3.3686
[Epoch 6/50] [Batch 100/157] Loss D: 0.7281 Loss G: 2.8709
[Epoch 7/50] [Batch 0/157] Loss D: 0.6579 Loss G: 4.0910
[Epoch 7/50] [Batch 100/157] Loss D: 0.7240 Loss G: 4.7313
[Epoch 8/50] [Batch 0/157] Loss D: 0.5791 Loss G: 2.8888
[Epoch 8/50] [Batch 100/157] Loss D: 1.2030 Loss G: 6.0587
[Epoch 9/50] [Batch 0/157] Loss

In [ ]:
# Final result
plt.figure(figsize=(10, 10))
plt.imshow(np.transpose(img_list[-1] if img_list else fake, (1, 2, 0)))
plt.axis("off")
plt.title("Synthetic Faces (500-image dataset)")
plt.savefig("synthetic_faces_500.png", dpi=150, bbox_inches='tight')
plt.show()

print("✅ Training complete. Generated image saved as 'synthetic_faces_500.png'")